# Improve a routing prompt

Improve instructions for routing support tickets and measure the resulting
labels against fixed answers. Adapt the instructions and labeled tickets to
your own task; the final section links to a real model integration.

This self-contained notebook lesson uses handwritten
simulations for both prompt revision and ticket responses. Scores describe
those simulations, not a live model. All code is on this page.

Start with instructions → propose a correction → test the ticket labels →
inspect what improved. The next lesson uses these same tickets to
[improve a reusable skill document](https://sentient-xyz.github.io/meta-evolve-docs/guides/skill-composition/) from mistakes.



<a id="define-what-correct-means"></a>
<a id="install-and-preview"></a>

## Install and define the task

Use a fresh notebook environment running **Python 3.12 or newer**.
Install directly from the published documentation:

In [ ]:
%pip install https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip

If you already imported Meta-Evolve, restart the kernel after installing.
Then run the remaining cells in order.

**Archived or offline docs:** use the ZIP included with that build. Put
`meta-evolve.zip` in the notebook's working folder (`%pwd` shows it; hosted
notebooks let you upload files), then run `%pip install ./meta-evolve.zip`
instead. Installing from source may still download build tools.

In [ ]:
import meta_evolve as meta

# Start with instructions that miss refunds and login problems.
ROUTING_SEED = "Route invoices to billing, passwords to account, otherwise technical."
# A fixed rewrite lets us try the loop without calling a model.
ROUTING_REVISION = (
    "Route invoices and refunds to billing, passwords and login to account, "
    "otherwise technical."
)

# Keep these expected teams fixed while we revise the instructions.
TICKETS = (
    ("Please send my invoice", "billing"),
    ("I need a refund", "billing"),
    ("I forgot my password", "account"),
    ("My login is blocked", "account"),
    ("The dashboard freezes", "technical"),
    ("Export is broken", "technical"),
)

These six development tickets contain one request each. Ambiguous requests,
priorities, and unseen tickets belong in a larger evaluation when you extend it.

<a id="inspect-the-evaluator"></a>

## Simulate the fixed responder

This small function stands in for a response-model call. It recognizes only the
keyword rules used in these two prompts. Its implementation stays fixed across
candidates; the prompt is the changing input.

In [ ]:
# This small simulation stands in for the agent that reads the instructions.
def routing_response(instructions, ticket):
    text = ticket.lower()
    # The simulated agent already handles these two topics.
    if "invoice" in text:
        return "billing"
    if "password" in text:
        return "account"
    # These topics need a matching word in the instruction document.
    if "refund" in instructions and "refund" in text:
        return "billing"
    if "login" in instructions and "login" in text:
        return "account"
    return "technical"  # Send anything else to the technical team.


def evaluate_routing(instructions):
    passed = 0
    # Ask the responder, then compare its answer with the expected team.
    for ticket, expected in TICKETS:
        actual = routing_response(instructions, ticket)
        if actual == expected:
            passed += 1
    return passed / len(TICKETS)  # Score = fraction of tickets routed correctly.

# Measure the starting instructions before trying a rewrite.
print(f"Starting score: {evaluate_routing(ROUTING_SEED):.0%}")
# Output:
# Starting score: 67%

Python owns the labels and scoring. The responder supplies an answer, not its
own grade. In a live integration, request failures must remain failed
measurements instead of being counted as wrong labels.

## Simulate the rewriting agent

The rewriting call has the same `agent(message)` shape as the parser lesson.
We use a different name so these cells can share a notebook with that lesson.

In [ ]:
def routing_agent(message):
    """A handwritten prompt revision, standing in for an SDK response."""
    # Read the current instructions from the message sent to the agent.
    instructions = message.rsplit("Current instructions:\n", 1)[1]
    if instructions in (ROUTING_SEED, ROUTING_REVISION):
        return ROUTING_REVISION  # Use our fixed rewrite for this simulation.
    raise ValueError("This simulation only recognizes the tutorial prompts.")


def propose_routing(instructions):
    # Give the rewriting agent the current instructions and a clear request.
    message = (
        "Improve the ticket-routing instructions. Return only instructions.\n"
        f"Current instructions:\n{instructions}"
    )
    return routing_agent(message)

<a id="assemble-the-improvement-loop"></a>
<a id="understand-the-result"></a>

## Run and inspect

In [ ]:
# Try one rewrite and keep whichever version scores better.
routing_result = meta.improve(
    seed=ROUTING_SEED,
    proposer=propose_routing,
    evaluator=evaluate_routing,
    trials=1,  # One revision, plus the starting version's evaluation.
)

# Compare the starting instructions (version 0) with the revision.
for number, attempt in enumerate(routing_result.trials()):
    print(f"Version {number}: {attempt.metrics['score']:.0%}")
# Use the selected instructions with your responder.
print(routing_result.best().value)
# Output:
# Version 0: 67%
# Version 1: 100%
# Route invoices and refunds to billing, passwords and login to account, otherwise technical.

The seed misses refunds and login requests; the revision handles both. These
fixed responses make the wiring easy to inspect. A live model may behave
differently even with the same instructions.

<a id="compare-with-the-parser"></a>

| Role | Parser | Routing prompt |
|---|---|---|
| Artifact / seed | Python source | Instructions |
| Proposer | Simulated source revision | Simulated prompt revision |
| Task / evaluator and objectives | Run Python and maximize passing fraction | Obtain responses and maximize correct labels |
| Search | Greedy, one revision in the quickstart | Greedy, one revision |
| Feedback / experience | Optional next layer | Omitted here; can be attached the same way |
| External limits | Fixed cases and attempt limit | Fixed responder, labels, and attempt limit |

## Change and predict

Set `trials=0` to measure only the initial instructions: 67%. Restore `trials=1`,
then add `("Please reimburse me", "billing")` to `TICKETS`. The simulated
responder does not understand that wording, so neither prompt solves it.
The best score is now 6/7. This is a limitation of the simulated responder,
not evidence about a real model.

<a id="complete-source"></a>
<a id="keep-the-work"></a>
<a id="rehearse-offline"></a>
<a id="run-with-a-model"></a>

For real requests, start with [Use your model SDK](https://sentient-xyz.github.io/meta-evolve-docs/guides/providers/), a
standalone notebook. The [larger routing example](https://sentient-xyz.github.io/meta-evolve-docs/guides/live-prompt/) adds
independent final checks. The
[feedback layer](https://sentient-xyz.github.io/meta-evolve-docs/learn/05-use-experience/) explains how recorded observations reach
a proposer; `improve` does not send them automatically.

[Next: Improve a support agent's skill](https://sentient-xyz.github.io/meta-evolve-docs/guides/skill-composition/) ·
[Save and reopen a run](https://sentient-xyz.github.io/meta-evolve-docs/learn/06-persist-run/)